# Sequential dilution, step 3 of 3 — merge into one ML-format table

Collects the recovered spectra for all eight SARS-CoV-2 variants into a single
CSV with one spectrum per row, labelled by variant and concentration, ready for
the downstream CNN.

**Input** — the recovered-spectrum folders written by step 2.

**Output** — one combined CSV (wavenumber columns, then `Label`, then `Conc`).

**Note** — earlier versions divided intensities by 400 here. They no longer do:
step 2 now writes spectra already on the mean-normalized scale.

**Next** — `../5_downstream_cnn/03_train_eval_recovered.ipynb`.


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
from tqdm import tqdm

In [ ]:
# Configuration
root_folder = "/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results"
output_folder = root_folder + "/03092026-combination_of_8_extracted_viruses_and_DNA"

# Create output directory
os.makedirs(output_folder, exist_ok=True)

folder_root_name = '03042026-test-Yingchuan-udpated_SDE_M7_extraction'

subfolder_name_list = [
    f'{folder_root_name}/B1',
    f'{folder_root_name}/B1351',
    f'{folder_root_name}/B16172',
    f'{folder_root_name}/BA5',
    f'{folder_root_name}/EG51',
    f'{folder_root_name}/JN1',
    f'{folder_root_name}/SARSCoV2',
    f'{folder_root_name}/XBB15',
    f'{folder_root_name}/ProbeDNA'
]

In [ ]:
# Code cell 2 (modified)
def extract_info_from_probeDNA_folder(folder_name):
    """
    Extract concentrations from ProbeDNA combination folder name.
    
    Example input: 'combination_1.0sg1_2.0sg1'
    Returns: conc_low='1.0', conc_high='2.0'
    """
    # Remove 'combination_' prefix
    parts = folder_name.replace('combination_', '').split('_')
    
    concentrations = []
    for part in parts:
        # Extract concentration before 'sg'
        if 'sg' in part:
            conc = part.split('sg')[0]
            try:
                concentrations.append(float(conc))
            except:
                continue
    
    if len(concentrations) >= 2:
        conc_low = str(min(concentrations))
        conc_high = str(max(concentrations))
    else:
        conc_low = conc_high = 'Unknown'
    
    return conc_low, conc_high


def extract_info_from_new_format_folder(folder_name):
    """
    Extract concentrations from new format folder name.
    
    Example input: 'conc_98.0_vs_195.0'
    Returns: conc_low='98.0', conc_high='195.0'
    """
    # Remove 'conc_' prefix and split by '_vs_'
    parts = folder_name.replace('conc_', '').split('_vs_')
    
    if len(parts) == 2:
        try:
            conc_low = str(float(parts[0]))
            conc_high = str(float(parts[1]))
            return conc_low, conc_high
        except:
            pass
    
    return 'Unknown', 'Unknown'


def collect_all_extracted_spectra(root_folder, subfolder_list, output_csv_path):
    """
    Collect all extracted spectra from multiple virus subfolders.
    Handles both old format (ProbeDNA) and new format (other viruses).
    
    Creates a large CSV with columns:
    - Wavenumber columns (400, 401, ..., 1799 or 1800)
    - Label: ['VirusName']
    - Conc: [higher_concentration]
    """
    
    all_spectra_data = []
    wavenumbers = None
    
    print("="*70)
    print("COLLECTING EXTRACTED SPECTRA FROM ALL VIRUSES")
    print("="*70)
    
    # First pass: count total folders to process
    total_folders = 0
    for subfolder_name in subfolder_list:
        subfolder_path = os.path.join(root_folder, subfolder_name)
        if os.path.exists(subfolder_path):
            # Count all subdirectories (both formats)
            subdirs = [d for d in os.listdir(subfolder_path) 
                      if os.path.isdir(os.path.join(subfolder_path, d))]
            total_folders += len(subdirs)
    
    print(f"\nTotal folders to process: {total_folders}\n")
    
    # Create progress bar
    pbar = tqdm(total=total_folders, desc="Processing folders", unit="folder")
    
    for subfolder_name in subfolder_list:
        subfolder_path = os.path.join(root_folder, subfolder_name)
        
        if not os.path.exists(subfolder_path):
            tqdm.write(f"\nWarning: Subfolder does not exist: {subfolder_path}")
            continue
        
        # Extract virus name from subfolder path
        virus_name = os.path.basename(subfolder_name)
        
        tqdm.write(f"\nProcessing: {virus_name}")
        
        # Determine format by checking if this is ProbeDNA
        is_probeDNA = (virus_name == 'ProbeDNA')
        
        # Get all subdirectories
        subdirs = [d for d in os.listdir(subfolder_path) 
                  if os.path.isdir(os.path.join(subfolder_path, d))]
        
        tqdm.write(f"  Found {len(subdirs)} subdirectories")
        tqdm.write(f"  Format: {'OLD (ProbeDNA)' if is_probeDNA else 'NEW'}")
        
        for subdir in subdirs:
            subdir_path = os.path.join(subfolder_path, subdir)
            
            try:
                if is_probeDNA:
                    # OLD FORMAT: combination_X.XsgY_Z.ZsgW/
                    # Find extracted_spectra_*.csv
                    csv_files = [f for f in os.listdir(subdir_path) 
                                if f.startswith('extracted_spectra_') and f.endswith('.csv')]
                    
                    if len(csv_files) == 0:
                        tqdm.write(f"    Warning: No extracted_spectra file in {subdir}")
                        pbar.update(1)
                        continue
                    
                    csv_file = csv_files[0]
                    csv_path = os.path.join(subdir_path, csv_file)
                    
                    # Read CSV
                    df = pd.read_csv(csv_path)
                    
                    # Get wavenumbers (first column)
                    if wavenumbers is None:
                        wavenumbers = df.iloc[:, 0].values
                        tqdm.write(f"  Wavenumber range: {wavenumbers[0]} to {wavenumbers[-1]}")
                        tqdm.write(f"  Number of wavenumber points: {len(wavenumbers)}")
                    
                    # Get final epoch (last column: Epoch_1000 or similar)
                    final_spectrum = df.iloc[:, -1].values
                    
                    # Extract concentrations from folder name
                    conc_low, conc_high = extract_info_from_probeDNA_folder(subdir)
                    
                    # Create row
                    row_data = {
                        'Label': [virus_name],
                        'Conc': [float(conc_high)]
                    }
                    
                    for i, wavenumber in enumerate(wavenumbers):
                        row_data[str(int(wavenumber))] = round(final_spectrum[i], 3)
                    
                    all_spectra_data.append(row_data)
                    
                else:
                    # NEW FORMAT: conc_X.X_vs_Y.Y/extracted_spectra_final_epoch.csv
                    csv_path = os.path.join(subdir_path, 'extracted_spectra_final_epoch.csv')
                    
                    if not os.path.exists(csv_path):
                        tqdm.write(f"    Warning: No extracted_spectra_final_epoch.csv in {subdir}")
                        pbar.update(1)
                        continue
                    
                    # Read CSV
                    df = pd.read_csv(csv_path)
                    
                    # Get wavenumbers (first column)
                    if wavenumbers is None:
                        wavenumbers = df.iloc[:, 0].values
                        tqdm.write(f"  Wavenumber range: {wavenumbers[0]} to {wavenumbers[-1]}")
                        tqdm.write(f"  Number of wavenumber points: {len(wavenumbers)}")
                    
                    # Extract concentrations from folder name
                    conc_low, conc_high = extract_info_from_new_format_folder(subdir)
                    
                    # Process each spectrum column (skip first column which is Wavenumbers)
                    for col in df.columns[1:]:
                        spectrum = df[col].values
                        
                        # Create row
                        row_data = {
                            'Label': [virus_name],
                            'Conc': [float(conc_high)]
                        }
                        
                        for i, wavenumber in enumerate(wavenumbers):
                            row_data[str(int(wavenumber))] = round(spectrum[i], 3)
                        
                        all_spectra_data.append(row_data)
                
                pbar.set_postfix({'virus': virus_name, 'folder': subdir[:30]})
                
            except Exception as e:
                tqdm.write(f"    Error processing {subdir}: {e}")
            
            pbar.update(1)
    
    pbar.close()
    
    # Create final DataFrame
    if len(all_spectra_data) == 0:
        print("\nError: No spectra were collected!")
        return None
    
    print(f"\n{'='*70}")
    print(f"Total spectra collected: {len(all_spectra_data)}")
    
    # Combine all rows
    final_df = pd.DataFrame(all_spectra_data)
    
    # Reorder columns: wavenumbers first, then Label, Conc
    metadata_cols = ['Label', 'Conc']
    wavenumber_cols = [str(int(w)) for w in wavenumbers]
    final_df = final_df[wavenumber_cols + metadata_cols]
    
    # Save to CSV
    final_df.to_csv(output_csv_path, index=False)
    print(f"\nSaved to: {output_csv_path}")
    print(f"DataFrame shape: {final_df.shape}")
    print(f"Column order: wavenumbers (400-1799), then Label, Conc")
    print(f"\nFirst few rows (showing last wavenumber and metadata):")
    display_cols = [wavenumber_cols[-1]] + metadata_cols
    print(final_df[display_cols].head(10))
    
    return final_df

In [ ]:
# Code cell 3 (unchanged)
# Execute
if __name__ == "__main__":
    output_csv_path = os.path.join(output_folder, "all_extracted_spectra_combined.csv")
    
    df_combined = collect_all_extracted_spectra(
        root_folder, 
        subfolder_name_list, 
        output_csv_path
    )
    
    if df_combined is not None:
        print("\n" + "="*70)
        print("COLLECTION COMPLETE!")
        print("="*70)
        print(f"\nSummary statistics:")
        print(f"  Total spectra: {len(df_combined)}")
        
        # Convert list columns to strings for counting
        virus_values = [v[0] if isinstance(v, list) else v for v in df_combined['Label']]
        print(f"  Unique viruses: {len(set(virus_values))}")
        print(f"\nVirus distribution:")
        from collections import Counter
        virus_counts = Counter(virus_values)
        for virus, count in sorted(virus_counts.items()):
            print(f"    {virus}: {count}")